# Carpet defect patch datasets

Set the paths and parameters, then run the cells you need.


In [ ]:

from pathlib import Path
import shutil, random
import numpy as np
from skimage.io import imread, imsave
from skimage.measure import label, regionprops

dataset = "carpet"
src = Path(f"datasets/{dataset}")
dst_random = Path(f"datasets/{dataset}-random-patch")
dst_grid = Path(f"datasets/{dataset}-grid-patch")
dst_centric = Path(f"datasets/{dataset}-centric-patchs")

patch = 224
seed = 1234
rng = random.Random(seed)
np.random.seed(seed)

def clear_dir(p):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def pairs(src):
    bad = src / "bad"
    masks = src / "bad-masks"
    out = []
    for img in sorted(bad.rglob("*.png")):
        rel = img.relative_to(bad)
        msk = masks / rel
        if msk.exists(): out.append((img, msk, rel.parent.as_posix(), rel))
    return out

def good_images(src):
    g = src / "good"
    return [(img, img.relative_to(g)) for img in sorted(g.rglob("*.png"))] if g.exists() else []

def mask01(msk):
    a = imread(msk)
    if a.ndim == 3: a = a[..., 0]
    return a > 0

def crop(a, y, x, s):
    return a[y:y+s, x:x+s]

def save_patch(a, out):
    out.parent.mkdir(parents=True, exist_ok=True)
    if a.dtype == bool: a = (a.astype(np.uint8) * 255)
    if a.dtype != np.uint8:
        a = np.clip(a, 0, 255)
        if a.max() <= 1: a = (a * 255).astype(np.uint8)
        else: a = a.astype(np.uint8)
    imsave(out.as_posix(), a, check_contrast=False)

def rand_xy(h, w, s, rng):
    if h < s or w < s: return None
    return rng.randint(0, h - s), rng.randint(0, w - s)

def grid_xy(h, w, s):
    if h < s or w < s: return []
    ys = list(range(0, h - s + 1, s))
    xs = list(range(0, w - s + 1, s))
    if ys[-1] != h - s: ys.append(h - s)
    if xs[-1] != w - s: xs.append(w - s)
    return [(y, x) for y in ys for x in xs]

def largest_box(msk):
    lab = label(msk)
    regs = regionprops(lab)
    if not regs: return None
    return max(regs, key=lambda r: r.area).bbox

def centered_start(c, s, m):
    if m <= s: return 0
    return max(0, min(int(round(c - s / 2)), m - s))

def best_grid_cell(msk, s):
    xy = grid_xy(*msk.shape, s)
    if not xy: return None
    vals = [crop(msk, y, x, s).sum() for y, x in xy]
    return xy[int(np.argmax(vals))]

def counts(root):
    d = {}
    bad = root / "bad"
    if bad.exists():
        for p in sorted([x for x in bad.iterdir() if x.is_dir()]): d[p.name] = len(list(p.rglob("*.png")))
    g = root / "good"
    d["good"] = len(list(g.rglob("*.png"))) if g.exists() else 0
    return d


In [4]:
def build_centric(src=src, dst=dst_centric, patch=patch, seed=seed):
    rng = random.Random(seed)
    clear_dir(dst)
    for img_p, msk_p, defect, rel in pairs(src):
        img = imread(img_p)
        msk = mask01(msk_p)
        box = largest_box(msk)
        if box is None: continue
        y0, x0, y1, x1 = box
        cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
        y, x = centered_start(cy, patch, msk.shape[0]), centered_start(cx, patch, msk.shape[1])
        save_patch(crop(img, y, x, patch), dst / "bad" / defect / rel.name)
        save_patch(crop(msk, y, x, patch), dst / "bad-masks" / defect / rel.name)
    for img_p, rel in good_images(src):
        img = imread(img_p)
        xy = rand_xy(*img.shape[:2], patch, rng)
        if xy is None: continue
        y, x = xy
        save_patch(crop(img, y, x, patch), dst / "good" / rel.name)

build_centric()

In [5]:
def build_random(src=src, dst=dst_random, patch=patch, seed=seed):
    rng = random.Random(seed)
    clear_dir(dst)
    for img_p, msk_p, defect, rel in pairs(src):
        img = imread(img_p)
        msk = mask01(msk_p)
        xy = rand_xy(*img.shape[:2], patch, rng)
        if xy is None: continue
        y, x = xy
        save_patch(crop(img, y, x, patch), dst / "bad" / defect / rel.name)
        save_patch(crop(msk, y, x, patch), dst / "bad-masks" / defect / rel.name)
    for img_p, rel in good_images(src):
        img = imread(img_p)
        xy = rand_xy(*img.shape[:2], patch, rng)
        if xy is None: continue
        y, x = xy
        save_patch(crop(img, y, x, patch), dst / "good" / rel.name)

build_random()


In [6]:

def build_grid(src=src, dst=dst_grid, patch=patch, seed=seed):
    rng = random.Random(seed)
    clear_dir(dst)
    for img_p, msk_p, defect, rel in pairs(src):
        img = imread(img_p)
        msk = mask01(msk_p)
        xy = best_grid_cell(msk, patch)
        if xy is None: continue
        y, x = xy
        save_patch(crop(img, y, x, patch), dst / "bad" / defect / rel.name)
        save_patch(crop(msk, y, x, patch), dst / "bad-masks" / defect / rel.name)
    for img_p, rel in good_images(src):
        img = imread(img_p)
        xy = grid_xy(*img.shape[:2], patch)
        if not xy: continue
        y, x = xy[rng.randrange(len(xy))]
        save_patch(crop(img, y, x, patch), dst / "good" / rel.name)

build_grid()
